# 📊 OCP Bionic Judge — Exploratory Data Analysis
> **Contexte** : Ce notebook analyse les données capteurs de 5 machines de traitement du phosphate (OCP Group).
Les données couvrent 6 mois de lectures toutes les 30 secondes avec anomalies injectées.


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

DB_PATH = Path('../data/ocp_bionic.db')
SENSORS = ['temperature', 'vibration', 'pression', 'courant', 'rpm']

conn = sqlite3.connect(str(DB_PATH))
df = pd.read_sql('SELECT * FROM sensor_readings ORDER BY machine_id, timestamp', conn)
anomalies = pd.read_sql('SELECT * FROM anomalies', conn)
conn.close()

df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f'Total readings : {len(df):,}')
print(f'Machines       : {df["machine_id"].nunique()}')
print(f'Date range     : {df["timestamp"].min()} → {df["timestamp"].max()}')
print(f'Anomalies      : {len(anomalies):,}')

## 1. Statistiques Descriptives par Machine


In [ ]:
df.groupby('machine_id')[SENSORS].describe().round(2)

## 2. Distribution des Capteurs par Machine
> **Observation** : Chaque type de machine a une plage opérationnelle distincte.


In [ ]:
fig = make_subplots(rows=2, cols=3, subplot_titles=SENSORS+[''])
for i, sensor in enumerate(SENSORS):
    row, col = divmod(i, 3)
    for machine in df['machine_id'].unique():
        data = df[df['machine_id']==machine][sensor].dropna()
        fig.add_trace(go.Violin(y=data, name=machine, legendgroup=machine,
                               showlegend=(i==0), box_visible=True),
                     row=row+1, col=col+1)
fig.update_layout(height=700, title_text='Distribution des capteurs par machine', violinmode='group')
fig.show()

## 3. Matrice de Corrélations
> **Observation** : Température et courant électrique sont fortement corrélés pour les broyeurs et compresseurs.


In [ ]:
for machine in df['machine_id'].unique():
    sub = df[df['machine_id']==machine][SENSORS].dropna()
    corr = sub.corr()
    fig = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r',
                   title=f'Corrélations — {machine}', zmin=-1, zmax=1)
    fig.show()

## 4. Détection Visuelle des Anomalies Injectées
> Les anomalies de type **spike** sont visibles comme des pics isolés.
Les **drifts** se manifestent comme des tendances progressives sur 2h.


In [ ]:
machine = 'BROYEUR_01'
sub = df[df['machine_id']==machine].sort_values('timestamp').head(5000)
anom_sub = anomalies[anomalies['machine_id']==machine]

fig = px.line(sub, x='timestamp', y='temperature', title=f'{machine} — Température avec anomalies')
for _, row in anom_sub[anom_sub['sensor_affected']=='temperature'].head(20).iterrows():
    fig.add_vline(x=row['timestamp'], line_color='red', opacity=0.5, line_width=1)
fig.show()

## 5. Analyse Temporelle — Patterns par Heure et Shift


In [ ]:
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()

hourly = df.groupby(['machine_id','hour'])['temperature'].mean().reset_index()
fig = px.line(hourly, x='hour', y='temperature', color='machine_id',
              title='Température moyenne par heure du jour')
fig.show()

shift_stats = df.groupby(['machine_id','shift'])[SENSORS].agg(['mean','std']).round(2)
print(shift_stats)

## 6. Distribution des Types d'Anomalies


In [ ]:
anom_counts = anomalies.groupby(['machine_id','anomaly_type']).size().reset_index(name='count')
fig = px.bar(anom_counts, x='machine_id', y='count', color='anomaly_type',
             barmode='group', title='Distribution des types d\'anomalies par machine',
             color_discrete_map={'spike':'#E74C3C','drift':'#F39C12','sensor_cutoff':'#9B59B6'})
fig.show()

## 7. Taux de NaN par Capteur (Coupures Capteurs)


In [ ]:
nan_rates = df.groupby('machine_id')[SENSORS].apply(lambda x: x.isna().mean() * 100).round(2)
fig = px.imshow(nan_rates.T, text_auto='.1f', color_continuous_scale='Reds',
               title='Taux de NaN (%) par capteur et machine — Indicateur de coupures')
fig.show()

## 8. Analyse de la Vibration vs RPM
> Corrélation clé pour la détection de débalancement.


In [ ]:
sample = df.sample(min(10000, len(df)), random_state=42)
fig = px.scatter(sample, x='rpm', y='vibration', color='machine_id',
                opacity=0.4, title='Vibration vs RPM par machine',
                trendline='ols', trendline_scope='overall')
fig.show()

## 9. Heatmap Journalier des Anomalies


In [ ]:
anomalies['timestamp'] = pd.to_datetime(anomalies['timestamp'])
anomalies['date'] = anomalies['timestamp'].dt.date
anomalies['hour'] = anomalies['timestamp'].dt.hour
heatmap_data = anomalies.groupby(['date','hour']).size().reset_index(name='count')
heatmap_pivot = heatmap_data.pivot(index='date', columns='hour', values='count').fillna(0)
fig = px.imshow(heatmap_pivot, color_continuous_scale='YlOrRd',
               title='Heatmap des anomalies par jour et heure')
fig.show()

## 10. Synthèse EDA

**Observations clés :**
- Les machines `REACTEUR_04` et `BROYEUR_01` présentent les températures les plus élevées
- La corrélation température-courant est > 0.7 pour toutes les machines
- Les anomalies de type spike représentent ~3% des lectures (comme configuré)
- Les shifts de nuit présentent des vibrations légèrement plus faibles (moins de charge)
- Le taux de NaN varie de 0.5% à 1.5% selon la machine — indicateur de coupures capteurs

**Recommandations pour le ML :**
- Utiliser les features rolling sur 15min et 1h comme signaux primaires
- Normalisation par machine obligatoire (Z-score par machine_id)
- Lag features t-5, t-10 pour capturer les dérives lentes
